## Concept Graph using ConceptNet

> **Note:** The original [Microsoft Concept Graph](https://concept.research.microsoft.com/) API is no longer available. This notebook has been updated to use [ConceptNet](https://conceptnet.io/) as a replacement, which is a freely available open knowledge graph with similar `is-a` relations between concepts.

[ConceptNet](https://conceptnet.io/) is a large semantic network of concepts with relationships such as `IsA`, `PartOf`, `UsedFor`, and more. It is available as:
 * A downloadable data file
 * A REST API (no API key required)

ConceptNet statistics:
 * Over 8 million nodes
 * 21+ million edges across 83 languages

## Using ConceptNet Web Service

[ConceptNet](https://conceptnet.io/) provides a REST API to explore `is-a` (IsA) relationships between concepts. No API key is required.
Here is the sample URL to call: `https://api.conceptnet.io/query?start=/c/en/microsoft&rel=/r/IsA&limit=10`

In [1]:
import urllib
import json

def http(x):
    response = urllib.request.urlopen(x)
    data = response.read()
    return data.decode('utf-8')

def query(x):
    concept = x.lower().replace(' ', '_')
    url = "https://api.conceptnet.io/query?start=/c/en/{}&rel=/r/IsA&limit=10".format(
        urllib.parse.quote(concept))
    try:
        result = json.loads(http(url))
    except Exception:
        return {}
    edges = result.get('edges', [])
    if not edges:
        return {}
    total_weight = sum(edge['weight'] for edge in edges)
    if total_weight == 0:
        return {}
    return {edge['end']['label']: edge['weight'] / total_weight for edge in edges}

query('microsoft')

{}

Let's try to categorize the news titles using parent concepts. To get news titles, we will use [NewsApi.org](http://newsapi.org) service. You need to obtain your own API key in order to use the service - go to the web site and register for free developer plan.

In [4]:
newsapi_key = 'b97e2d1bf1404b148b23cc062aebaa78'
def get_news(country='us'):
    res = json.loads(http("https://newsapi.org/v2/top-headlines?country={0}&apiKey={1}".format(country,newsapi_key)))
    return res['articles']

all_titles = [x['title'] for x in get_news('us')+get_news('gb')]

In [5]:
all_titles

["Swae Lee's Coachella's Set Reportedly Cut Off Before 'Black Beatles' - TMZ",
 'Mamdani’s administration scrambles on key appointment - Politico',
 'Live Updates: U.S. official says no agreements have been made in Iran peace talks yet as shaky ceasefire holds - CBS News',
 'RFK Jr. has turned corporate America’s name to mud, POLITICO Poll finds - Politico',
 'India cracks down on satirists for turning its prime minister into a punch line - NPR',
 "Gut troubles? This gastroenterologist has tips to help you achieve 'poophoria' - NPR",
 'Woman Diagnosed with Vulvar, Cervical and Anal Cancer After Learning Her Husband of 30 Years Had Cheated on Her - Yahoo Lifestyle Singapore',
 'An Orban loss in Hungary’s election could be the turning point Putin fears - France 24',
 "Peru's election: A battle for the Presidency amid political chaos and crime - NPR",
 'How AI is getting better at finding security holes - NPR',
 'Anthropic’s latest AI model strikes fear into banks - Morning Brew',
 "Marri

First of all, we want to be able to extract nouns from news titles. We will use `TextBlob` library to do this, which simplifies a lot of typical NLP tasks like this.

In [6]:
import sys
!{sys.executable} -m pip install textblob
!{sys.executable} -m textblob.download_corpora
from textblob import TextBlob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.3/624.3 kB 1.5 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 996.7 kB/s  0:00:01ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [textblob]4/6 [nltk]b]
[nltk_data] Downloading package brown to
[nltk_data]     /Users/thanhphatchau/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/thanhphatchau/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/thanhphatchau/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/thanhphatchau/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to
[nltk_data]     /Users/thanhphatchau/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to
[nltk_data

In [7]:
w = {}
for x in all_titles:
    for n in TextBlob(x).noun_phrases:
        if n in w:
            w[n].append(x)
        else:
            w[n]=[x]
{ x:len(w[x]) for x in w.keys()}

{'swae lee': 1,
 'coachella': 1,
 'set reportedly cut off': 1,
 'beatles': 1,
 'tmz': 1,
 'mamdani': 1,
 '’ s administration scrambles': 1,
 'key appointment': 1,
 'politico': 2,
 'live updates': 1,
 'u.s.': 1,
 'iran': 2,
 'shaky ceasefire': 1,
 'cbs news': 1,
 'rfk jr.': 1,
 'america': 1,
 '’ s name': 1,
 'politico poll': 1,
 'india': 1,
 'prime minister': 1,
 'punch line': 1,
 'npr': 4,
 'gut': 1,
 'woman diagnosed': 1,
 'vulvar': 1,
 'cervical': 1,
 'anal': 1,
 'learning': 1,
 'husband': 1,
 'years': 1,
 'cheated': 1,
 'yahoo lifestyle singapore': 1,
 'orban': 1,
 'hungary': 1,
 '’ s election': 1,
 'turning point': 1,
 'putin': 1,
 'france': 1,
 'peru': 1,
 "'s election": 1,
 'presidency': 1,
 'political chaos': 1,
 'ai': 2,
 'security holes': 1,
 'anthropic': 1,
 '’ s': 2,
 'model strikes': 1,
 'morning brew': 1,
 'marriage status': 1,
 'surprising link': 1,
 'cancer risk': 1,
 'study suggests': 1,
 "'clear signal": 1,
 'aol.com': 1,
 "'affordability economy": 1,
 'prices': 1,
 's

We can see that nouns do not give us large thematic groups. Let's substitute nouns by more general terms obtained from the concept graph. This will take some time, because we are doing REST call for each noun phrase.

In [13]:
w = {}
for x in all_titles:
    for noun in TextBlob(x).noun_phrases:
        terms = query(noun)
        for term in [u for u in terms.keys() if terms[u]>0.1]:
            if term in w:
                w[term].append(x)
            else:
                w[term]=[x]

In [11]:
{ x:len(w[x]) for x in w.keys() if len(w[x])>3}

{}

In [12]:
print('\nECONOMY:\n'+'\n'.join(w['economy']))
print('\nNATION:\n'+'\n'.join(w['nation']))
print('\nPERSON:\n'+'\n'.join(w['person']))

KeyError: 'economy'